## 1. Setup & Environment

In [ ]:
import os
import zipfile
if not os.path.exists('./Student folder') and not os.path.exists('../Student folder'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    if os.path.exists('/content/drive/MyDrive/student_folder.zip') and not os.path.exists('/content/Student folder'):
        with zipfile.ZipFile('/content/drive/MyDrive/student_folder.zip', 'r') as zf:
            zf.extractall('/content')

In [ ]:
import glob
import math
import time
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss, confusion_matrix

In [ ]:
USE_GPU = False
try:
    from cuml.neighbors import KNeighborsClassifier as cuKNeighborsClassifier
    _probe = cuKNeighborsClassifier(n_neighbors=3, weights='distance')
    _probe.fit(np.zeros((6, 2), dtype=np.float32), np.array([0, 1, 0, 1, 0, 1]))
    USE_GPU = True
except Exception:
    USE_GPU = False

def to_numpy(arr):
    if hasattr(arr, 'to_numpy'):
        return arr.to_numpy()
    if hasattr(arr, 'get'):
        return arr.get()
    return np.asarray(arr)

def make_knn(n_neighbors):
    if USE_GPU:
        return cuKNeighborsClassifier(n_neighbors=n_neighbors, weights='distance')
    return KNeighborsClassifier(n_neighbors=n_neighbors, weights='distance', algorithm='kd_tree', leaf_size=40, n_jobs=-1)

print('Backend:', 'GPU (cuML)' if USE_GPU else 'CPU (scikit-learn KDTree)')

In [ ]:
DATA_DIR = './Student folder'
for cand in ['/content/Student folder', '/content/drive/MyDrive/Student Folder', './Student folder', '../Student folder']:
    if os.path.exists(cand):
        DATA_DIR = cand
        break
if os.path.isdir(os.path.join(DATA_DIR, 'Student folder')):
    DATA_DIR = os.path.join(DATA_DIR, 'Student folder')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'student_test')
print('Data dir:', DATA_DIR)
print('Train exists:', os.path.exists(TRAIN_DIR), '| Test exists:', os.path.exists(TEST_DIR))

## 2. Team Preprocessing & Feature Engineering

In [ ]:
FEATURE_COLS = [
    'dist_orthogonal',
    'proj_ratio',
    'dist_to_start',
    'dist_to_end',
    'distance_rank',
    'road_crowding_25_50m',
    'arc_length',
    'distance_travelled',
    'heading_diff',
    'heading_cos',
    'speed',
    'time_gap',
    'is_time_gap'
]

def load_trip_data(trip_dir):
    tid = os.path.basename(trip_dir)
    nodes = np.atleast_2d(np.loadtxt(os.path.join(trip_dir, f'{tid}.nodes'), dtype=np.float64))
    arcs = np.atleast_2d(np.loadtxt(os.path.join(trip_dir, f'{tid}.arcs'), dtype=np.int32))
    track = np.atleast_2d(np.loadtxt(os.path.join(trip_dir, f'{tid}.track'), dtype=np.float64))
    route_path = os.path.join(trip_dir, f'{tid}.route')
    route = np.atleast_1d(np.loadtxt(route_path, dtype=np.int32)).tolist() if os.path.exists(route_path) else None
    return tid, nodes, arcs, track, route

def clean_gps_track(track, gap_threshold=3.0):
    if len(track) == 0:
        return track, np.array([]), np.array([])
    valid = (
        np.isfinite(track).all(axis=1) &
        (track[:, 0] >= -180.0) & (track[:, 0] <= 180.0) &
        (track[:, 1] >= -90.0) & (track[:, 1] <= 90.0)
    )
    cleaned = track[valid]
    cleaned = cleaned[np.argsort(cleaned[:, 2])]
    _, uniq = np.unique(cleaned[:, 2], return_index=True)
    cleaned = cleaned[np.sort(uniq)]
    n_pts = len(cleaned)
    dt = np.ones(n_pts)
    if n_pts > 1:
        dt[1:] = np.diff(cleaned[:, 2])
    dt = np.where(dt <= 0, 1.0, dt)
    is_time_gap = (dt > gap_threshold).astype(np.int32)
    return cleaned, dt, is_time_gap

def match_track_to_route(track_m, u_m, v_m, route):
    n_pts = len(track_m)
    n_route = len(route)
    true_arcs = np.empty(n_pts, dtype=np.int32)
    r_idx = 0
    for i in range(n_pts):
        p = track_m[i]
        best_d = 1e9
        best_r = r_idx
        best_arc = route[r_idx]
        for r in range(max(0, r_idx - 2), min(n_route, r_idx + 15)):
            a = route[r]
            u, v = u_m[a], v_m[a]
            seg = v - u
            l2 = np.dot(seg, seg)
            if l2 == 0:
                d = np.linalg.norm(p - u)
            else:
                t = np.clip(np.dot(p - u, seg) / l2, 0.0, 1.0)
                d = np.linalg.norm(p - (u + t * seg))
            if d < best_d:
                best_d = d
                best_r = r
                best_arc = a
        r_idx = best_r
        true_arcs[i] = best_arc
    return true_arcs

In [ ]:
def extract_trip_features(trip_dir, search_radius=60.0, max_cands_per_pt=15):
    tid, nodes, arcs, raw_track, route = load_trip_data(trip_dir)
    track, dt, is_time_gap = clean_gps_track(raw_track, gap_threshold=3.0)
    n_pts = len(track)
    if n_pts == 0:
        return pd.DataFrame(), 0.0, arcs
    lat_mean = np.mean(track[:, 1])
    kx = 111320.0 * math.cos(math.radians(lat_mean))
    ky = 110540.0
    nodes_m = np.column_stack([nodes[:, 0] * kx, nodes[:, 1] * ky])
    track_m = np.column_stack([track[:, 0] * kx, track[:, 1] * ky])
    u_m = nodes_m[arcs[:, 0]]
    v_m = nodes_m[arcs[:, 1]]
    arc_vec = v_m - u_m
    arc_len = np.linalg.norm(arc_vec, axis=1)
    true_arcs = match_track_to_route(track_m, u_m, v_m, route) if route is not None else None
    sample_pts = np.vstack([u_m + frac * arc_vec for frac in [0.0, 0.25, 0.5, 0.75, 1.0]])
    sample_arc_ids = np.tile(np.arange(len(arcs)), 5)
    tree = cKDTree(sample_pts)
    query_res = tree.query_ball_point(track_m, r=search_radius)
    d_pos = np.zeros((n_pts, 2))
    d_pos[1:] = np.diff(track_m, axis=0)
    distance_travelled = np.linalg.norm(d_pos, axis=1)
    speed = distance_travelled / dt
    traj_heading = np.arctan2(d_pos[:, 1], d_pos[:, 0])
    if n_pts > 1:
        traj_heading[0] = traj_heading[1]
    arc_heading = np.arctan2(arc_vec[:, 1], arc_vec[:, 0])
    rows = []
    miss_count = 0
    for i in range(n_pts):
        p = track_m[i]
        cands = set(sample_arc_ids[query_res[i]])
        if len(cands) < 3:
            _, near_idx = tree.query(p, k=min(5, len(sample_pts)))
            cands.update(np.atleast_1d(sample_arc_ids[near_idx]).tolist())
        cands = list(cands)
        if true_arcs is not None:
            t_arc = true_arcs[i]
            if t_arc not in cands:
                miss_count += 1
                cands.append(t_arc)
        cand_arr = np.array(cands, dtype=np.int32)
        u_cand = u_m[cand_arr]
        v_cand = v_m[cand_arr]
        vec_cand = arc_vec[cand_arr]
        len_cand = arc_len[cand_arr]
        len_sq = len_cand ** 2
        p_u = p - u_cand
        t_proj = np.where(len_sq > 0, np.clip(np.sum(p_u * vec_cand, axis=1) / np.maximum(len_sq, 1e-6), 0.0, 1.0), 0.0)
        proj_pts = u_cand + t_proj[:, None] * vec_cand
        dist_ortho = np.linalg.norm(p - proj_pts, axis=1)
        dist_start = np.linalg.norm(p_u, axis=1)
        dist_end = np.linalg.norm(p - v_cand, axis=1)
        theta_diff = np.abs(traj_heading[i] - arc_heading[cand_arr])
        theta_diff = np.where(theta_diff > np.pi, 2 * np.pi - theta_diff, theta_diff)
        cos_theta = np.cos(theta_diff)
        sort_order = np.argsort(dist_ortho)
        if len(sort_order) > max_cands_per_pt:
            keep = list(sort_order[:max_cands_per_pt])
            if true_arcs is not None:
                t_pos = np.where(cand_arr == true_arcs[i])[0]
                if len(t_pos) > 0 and t_pos[0] not in keep:
                    keep[-1] = t_pos[0]
            sort_order = np.array(keep)
        sel_dist = dist_ortho[sort_order]
        sel_rank = np.empty(len(sort_order), dtype=np.int32)
        sel_rank[np.argsort(sel_dist, kind='stable')] = np.arange(1, len(sort_order) + 1)
        crowding = np.array([
            np.count_nonzero((sel_dist >= 25.0) & (sel_dist <= 50.0) & (np.arange(len(sel_dist)) != k))
            for k in range(len(sel_dist))
        ], dtype=np.int32)
        for pos, idx in enumerate(sort_order):
            a_id = cand_arr[idx]
            rows.append({
                'trip_id': tid,
                'obs_idx': i,
                'arc_id': int(a_id),
                'dist_orthogonal': float(dist_ortho[idx]),
                'proj_ratio': float(t_proj[idx]),
                'dist_to_start': float(dist_start[idx]),
                'dist_to_end': float(dist_end[idx]),
                'distance_rank': int(sel_rank[pos]),
                'road_crowding_25_50m': int(crowding[pos]),
                'arc_length': float(len_cand[idx]),
                'distance_travelled': float(distance_travelled[i]),
                'heading_diff': float(theta_diff[idx]),
                'heading_cos': float(cos_theta[idx]),
                'speed': float(speed[i]),
                'time_gap': float(dt[i]),
                'is_time_gap': int(is_time_gap[i]),
                'is_true_arc': 1 if (true_arcs is not None and a_id == true_arcs[i]) else 0
            })
    miss_rate = (miss_count / n_pts) if n_pts > 0 else 0.0
    return pd.DataFrame(rows), miss_rate, arcs

def build_split(folder_path):
    dfs, misses, arcs_dict = [], [], {}
    for tdir in sorted(glob.glob(os.path.join(folder_path, '*'))):
        if not os.path.isdir(tdir):
            continue
        df, m_rate, arcs = extract_trip_features(tdir)
        if not df.empty:
            dfs.append(df)
            misses.append(m_rate)
            arcs_dict[os.path.basename(tdir)] = arcs
    return pd.concat(dfs, ignore_index=True), float(np.mean(misses)), arcs_dict

In [ ]:
t0 = time.time()
train_df, train_miss, train_arcs = build_split(TRAIN_DIR)
test_df, test_miss, test_arcs = build_split(TEST_DIR)
print('Feature build time (s):', round(time.time() - t0, 2))
print('Train:', train_df.shape, '| Miss rate:', round(train_miss, 4), '| Pos ratio:', round(train_df['is_true_arc'].mean(), 4))
print('Test: ', test_df.shape, '| Miss rate:', round(test_miss, 4), '| Pos ratio:', round(test_df['is_true_arc'].mean(), 4))

## 3. GroupKFold CV & KNN Tuning

In [ ]:
def evaluate_predictions(df, y_prob, arcs_dict):
    y_true = df['is_true_arc'].to_numpy()
    y_pred = (y_prob >= 0.5).astype(np.int32)
    tmp = df[['trip_id', 'obs_idx', 'arc_id', 'is_true_arc']].copy()
    tmp['prob'] = y_prob
    idx_max = tmp.groupby(['trip_id', 'obs_idx'])['prob'].idxmax()
    top1 = tmp.loc[idx_max].sort_values(['trip_id', 'obs_idx'])
    top1_acc = float(top1['is_true_arc'].mean())
    total_t, valid_t = 0, 0
    for tid, grp in top1.groupby('trip_id'):
        seq = grp['arc_id'].to_numpy()
        arcs = arcs_dict.get(tid)
        if arcs is None or len(seq) < 2:
            continue
        for i in range(len(seq) - 1):
            total_t += 1
            if seq[i] == seq[i + 1] or arcs[seq[i], 1] == arcs[seq[i + 1], 0]:
                valid_t += 1
    return {
        'cand_acc': round(float(accuracy_score(y_true, y_pred)), 4),
        'precision': round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        'recall': round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        'macro_f1': round(float(f1_score(y_true, y_pred, average='macro')), 4),
        'log_loss': round(float(log_loss(y_true, y_prob, labels=[0, 1])), 4),
        'top1_acc': round(top1_acc, 4),
        'route_continuity': round(valid_t / max(total_t, 1), 4),
        'cm': confusion_matrix(y_true, y_pred)
    }

X_train_raw = train_df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
X_test_raw = test_df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
y_train = train_df['is_true_arc'].to_numpy(dtype=np.int32)
groups = train_df['trip_id'].to_numpy()

gkf = GroupKFold(n_splits=5)
cv_results = []
for k in [7, 15, 25]:
    fold_f1, fold_top1, fold_ll = [], [], []
    t_cv = time.time()
    for tr_idx, va_idx in gkf.split(X_train_raw, y_train, groups=groups):
        med = X_train_raw.iloc[tr_idx].median().fillna(0.0)
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train_raw.iloc[tr_idx].fillna(med)).astype(np.float32)
        X_va = scaler.transform(X_train_raw.iloc[va_idx].fillna(med)).astype(np.float32)
        knn_cv = make_knn(k)
        knn_cv.fit(X_tr, y_train[tr_idx])
        probs = to_numpy(knn_cv.predict_proba(X_va))[:, 1]
        m = evaluate_predictions(train_df.iloc[va_idx], probs, train_arcs)
        fold_f1.append(m['macro_f1'])
        fold_top1.append(m['top1_acc'])
        fold_ll.append(m['log_loss'])
    row = {
        'n_neighbors': k,
        'cv_macro_f1': round(float(np.mean(fold_f1)), 4),
        'cv_top1_acc': round(float(np.mean(fold_top1)), 4),
        'cv_log_loss': round(float(np.mean(fold_ll)), 4),
        'cv_time_s': round(time.time() - t_cv, 2)
    }
    cv_results.append(row)
    print(row)

best_k = max(cv_results, key=lambda r: r['cv_macro_f1'])['n_neighbors']
print('Selected n_neighbors:', best_k)

## 4. Final KNN Training & Student-Test Evaluation

In [ ]:
med_full = X_train_raw.median().fillna(0.0)
scaler_full = StandardScaler()
X_train_scaled = scaler_full.fit_transform(X_train_raw.fillna(med_full)).astype(np.float32)
X_test_scaled = scaler_full.transform(X_test_raw.fillna(med_full)).astype(np.float32)

knn = make_knn(best_k)
t0 = time.time()
knn.fit(X_train_scaled, y_train)
train_time = time.time() - t0

t0 = time.time()
y_prob = to_numpy(knn.predict_proba(X_test_scaled))[:, 1]
inf_time = time.time() - t0

res = evaluate_predictions(test_df, y_prob, test_arcs)
print('n_neighbors:', best_k)
print('Training Time (s):', round(train_time, 4))
print('Inference Time (s):', round(inf_time, 4))
print('Candidate Accuracy:', res['cand_acc'])
print('Precision:', res['precision'])
print('Recall:', res['recall'])
print('Student-Test Macro F1:', res['macro_f1'])
print('Log Loss:', res['log_loss'])
print('Observation-level Top-1 Accuracy:', res['top1_acc'])
print('Candidate Miss Rate:', round(test_miss, 4))
print('Route Continuity:', res['route_continuity'])
print('Confusion Matrix:\n', res['cm'])